# Finish the defence sweep -- re-run ONLY `atk_fedavg` (Kaggle)

The defence sweep's `atk_fedavg` run crashed: `model_replacement` diverged plain FedAvg
to NaN and the old AUC metric errored. The metric is now NaN-safe, so this quick
notebook re-runs just that one variant (~15 min) and gives you its result to download.

Same setup as the defence sweep (6 nodes, 128px, ~12k images, 8 rounds). Sidebar:
**GPU On, Internet On**, and **+ Add Input** `nih-chest-xrays/data` (+ optionally the
224px mirror). Run All, then download the zip at the end and send it back.


## 1. GPU check

In [ ]:
import torch, sys
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 2. Install deps + clone repo (needs the LATEST pushed code + Internet On)

In [ ]:
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/kaggle/working/fl"):
    !git clone -q $REPO_URL /kaggle/working/fl
%cd /kaggle/working/fl
!git log --oneline -1   # must be the commit that includes the NaN-safe metric + resume

## 3. Locate the mounted dataset

In [ ]:
import glob, os, shutil
import pandas as pd
NIH = "/kaggle/working/nih"
os.makedirs(NIH, exist_ok=True)
print("Contents of /kaggle/input:")
for d in sorted(glob.glob("/kaggle/input/*")):
    pngs = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    print(f"  {d}  | {pngs} png")
all_csv = glob.glob("/kaggle/input/**/*.csv", recursive=True)
labels = next((c for c in all_csv if os.path.basename(c).lower() == "data_entry_2017.csv"), None)
if labels is None:
    for c in all_csv:
        try:
            if {"Image Index", "Finding Labels"}.issubset(pd.read_csv(c, nrows=1).columns):
                labels = c; break
        except Exception:
            pass
assert labels, "No labels CSV. '+ Add Input' -> 'nih-chest-xrays/data'."
shutil.copy(labels, f"{NIH}/Data_Entry_2017.csv")
bb = next((c for c in all_csv if os.path.basename(c).lower() == "bbox_list_2017.csv"), None)
if bb: shutil.copy(bb, f"{NIH}/BBox_List_2017.csv")
best, best_n = None, 0
for d in sorted(glob.glob("/kaggle/input/*")):
    n = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    if n > best_n: best, best_n = d, n
IMAGE_ROOT = best
print("IMAGE_ROOT:", IMAGE_ROOT, "| png:", best_n)
assert best_n > 0, "No PNG images in any mounted input."

## 4. Run ONLY the `atk_fedavg` variant

Identical config to the full defence sweep, so the partition/seed match and this run
slots straight into the existing results.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/fl")
from src.experiments.config import load_config
from src.experiments.sweep import run_sweep, default_defence_variants

cfg = load_config("configs/kaggle_defences.yaml")
cfg["data"]["root"] = NIH
cfg["data"]["image_root"] = IMAGE_ROOT
cfg["partition"]["manifest_path"] = f"{NIH}/manifest_def.json"
cfg["data"]["subsample"]["max_images"] = 12000   # MUST match your sweep run
cfg["training"]["rounds"] = 8                     # MUST match your sweep run

variants = [v for v in default_defence_variants(attack_type="model_replacement",
            attack_fraction=0.34, dp_noise=(1.0, 2.0), robust_rule="median")
            if v["name"] == "atk_fedavg"]
OUT = "/kaggle/working/results/defences"
sweep = run_sweep(cfg, variants, OUT, data_root=NIH, resume=False)
auc = sweep["runs"][0].get("mean_auc")
print("\n>>> atk_fedavg final mean AUC =", auc, "(expected ~0.50 — FedAvg diverges under the attack)")

## 5. Download the result

Downloads `atk_fedavg.zip` (contains `atk_fedavg/federated_results.json`). Send this
file back, or just report the printed AUC above.

In [ ]:
import shutil
from IPython.display import FileLink, display
shutil.make_archive("/kaggle/working/atk_fedavg", "zip", "/kaggle/working/results/defences/atk_fedavg")
print("zipped -> /kaggle/working/atk_fedavg.zip")
display(FileLink("atk_fedavg.zip"))